## Импорт библиотек

In [1]:
import pandas as pd
import string
import re
import nltk
import pymorphy3

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
from warnings import filterwarnings
filterwarnings('ignore')

In [3]:
df = pd.read_csv("parse_data.csv").drop('Unnamed: 0', axis=1)

In [4]:
df.head()

,title,description,release_date,tags,rating
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         249 non-null    object
 1   description   249 non-null    object
 2   release_date  249 non-null    object
 3   tags          249 non-null    object
 4   rating        249 non-null    object
dtypes: object(5)
memory usage: 9.9+ KB


## Предварительная обработка данных

In [6]:
def remove_punctuation(text): 
    return "".join([ch if ch not in string.punctuation else ' ' for ch in text])

def remove_numbers(text): 
    return ''.join([i if not i.isdigit() else ' ' for i in text])

def remove_multiple_spaces(text): 
    return re.sub(r'\s+', ' ', text, flags=re.I)

st = '❯ —«»'
def remove_othersymbol(text):
    return ''.join([ch if ch not in st else ' ' for ch in text])

In [7]:
df['prep_text'] = [remove_multiple_spaces(remove_numbers(remove_othersymbol(remove_punctuation(text.lower())))) for text in df['description']]

In [8]:
df.head()

,title,description,release_date,tags,rating,prep_text
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...


In [9]:
russian_stopwords = stopwords.words("russian") 

In [10]:
russian_stopwords.extend(['т.д.', 'т', 'д', 'это','который','с','своём','всем','наш', 'свой', 'фильм']) 

In [11]:
def tokenize(text):
    t = word_tokenize(text)
    tokens = [token for token in t if token not in russian_stopwords]
    text = " ".join(tokens)
    return text

In [12]:
df['tokenize_text'] = [tokenize(text) for text in df['prep_text']]

In [13]:
df.head()

,title,description,release_date,tags,rating,prep_text,tokenize_text
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...,побег шоушенка считается одним лучших истории ...
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...,ставший классикой своего жанра рассказывающий ...
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...,бэтман вершит правосудие готэме партнерами ста...
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...,продолжение эпохальной саги режиссера френсиса...
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...,присяжных деле убийстве пытается убедить остал...


In [14]:
stemmer = SnowballStemmer("russian")

stem_list = []
for text in (df['tokenize_text']):
    try:
        tokens = word_tokenize(text)
        res = list()
        for word in tokens:
            p = stemmer.stem(word)
            res.append(p)
        text = " ".join(res)
        stem_list.append(text)
    except Exception as e:
        print(e)
        
df['text_stem'] = stem_list

In [15]:
df.head()

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...,побег шоушенка считается одним лучших истории ...,побег шоушенк счита одн лучш истор кин режиссе...
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...,ставший классикой своего жанра рассказывающий ...,ставш классик сво жанр рассказыва истор влияте...
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...,бэтман вершит правосудие готэме партнерами ста...,бэтма верш правосуд готэм партнер станов инспе...
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...,продолжение эпохальной саги режиссера френсиса...,продолжен эпохальн саг режиссер френсис форд к...
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...,присяжных деле убийстве пытается убедить остал...,присяжн дел убийств пыта убед остальн присяжн ...


In [16]:
morph = pymorphy3.MorphAnalyzer(lang='ru')

In [17]:
%%time
lemm_texts_list = []
for text in (df['tokenize_text']):
    try:
        tokens = word_tokenize(text)
        res = list()
        for word in tokens:
            p = morph.parse(word)[0]
            res.append(p.normal_form)
        text = " ".join(res)
        lemm_texts_list.append(text)
    except Exception as e:
        print(e)
    
df['text_lemm'] = lemm_texts_list

CPU times: total: 2.69 s
Wall time: 2.75 s


In [18]:
df.head()

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...,побег шоушенка считается одним лучших истории ...,побег шоушенк счита одн лучш истор кин режиссе...,побег шоушенка считаться один хороший история ...
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...,ставший классикой своего жанра рассказывающий ...,ставш классик сво жанр рассказыва истор влияте...,стать классика свой жанр рассказывать история ...
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...,бэтман вершит правосудие готэме партнерами ста...,бэтма верш правосуд готэм партнер станов инспе...,бэтман вершить правосудие готэма партнёр стано...
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...,продолжение эпохальной саги режиссера френсиса...,продолжен эпохальн саг режиссер френсис форд к...,продолжение эпохальный сага режиссёр френсиса ...
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...,присяжных деле убийстве пытается убедить остал...,присяжн дел убийств пыта убед остальн присяжн ...,присяжный дело убийство пытаться убедить остал...


In [19]:
df['text_lemm'] = [tokenize(text) for text in df['text_lemm']]

In [20]:
df.head()

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...,побег шоушенка считается одним лучших истории ...,побег шоушенк счита одн лучш истор кин режиссе...,побег шоушенка считаться хороший история кино ...
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...,ставший классикой своего жанра рассказывающий ...,ставш классик сво жанр рассказыва истор влияте...,стать классика жанр рассказывать история влият...
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...,бэтман вершит правосудие готэме партнерами ста...,бэтма верш правосуд готэм партнер станов инспе...,бэтман вершить правосудие готэма партнёр стано...
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...,продолжение эпохальной саги режиссера френсиса...,продолжен эпохальн саг режиссер френсис форд к...,продолжение эпохальный сага режиссёр френсиса ...
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...,присяжных деле убийстве пытается убедить остал...,присяжн дел убийств пыта убед остальн присяжн ...,присяжный дело убийство пытаться убедить остал...


## Векторизация текстовых данных

In [21]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=100000990000000000000,
                                 min_df=0.01, stop_words=russian_stopwords,
                                 ngram_range=(1,3))

In [22]:
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text_lemm'])

In [23]:
tfidf_matrix.shape

(249, 659)

## Тематическое моделирование

In [24]:
from sklearn.decomposition import TruncatedSVD

In [25]:
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: жизнь, год, история, человек, стать, война, друг, время, город, хороший
Topic 1: война, мировой, мировой война, второй мировой война, второй мировой, второй, рассказывать, американский, история, год
Topic 2: история, год, стать, хороший, реальный, жизнь, рассказывать, реальный история, работать, зритель
Topic 3: дело, убийство, друг, человек, расследование, местный, решать, женщина, полицейский, расследование дело
Topic 4: дело, жизнь, убийство, стать, жена, работа, дочь, решать, полицейский, фрэнк


In [26]:
from sklearn.decomposition import NMF

In [27]:
# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: история, год, хороший, реальный, рассказывать, стать, американский, реальный история, время, человек
Topic 1: война, мировой война, мировой, второй, второй мировой, второй мировой война, еврей, время второй мировой, время второй, ужас
Topic 2: друг, город, человек, поиск, новый, жить, отправляться, найти, вместе, день
Topic 3: дело, убийство, полицейский, женщина, расследование, расследование дело, решать, расследовать, убийца, местный
Topic 4: жизнь, работа, стать, ребёнок, депрессия, отношение, деньга, семья, помогать, однако


## Кластеризация

In [28]:
num_clusters = 5

# Метод к-средних - KMeans
from sklearn.cluster import KMeans
km = KMeans(n_clusters=num_clusters, random_state=0)

In [29]:
km.fit(tfidf_matrix)

  File "C:\Users\robhul\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\robhul\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\robhul\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\robhul\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


KMeans(n_clusters=5, random_state=0)

In [30]:
clusterkm = km.labels_.tolist()
df['cluster']= clusterkm

In [31]:
df.head()

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...,побег шоушенка считается одним лучших истории ...,побег шоушенк счита одн лучш истор кин режиссе...,побег шоушенка считаться хороший история кино ...,2
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...,ставший классикой своего жанра рассказывающий ...,ставш классик сво жанр рассказыва истор влияте...,стать классика жанр рассказывать история влият...,4
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...,бэтман вершит правосудие готэме партнерами ста...,бэтма верш правосуд готэм партнер станов инспе...,бэтман вершить правосудие готэма партнёр стано...,0
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...,продолжение эпохальной саги режиссера френсиса...,продолжен эпохальн саг режиссер френсис форд к...,продолжение эпохальный сага режиссёр френсиса ...,4
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...,присяжных деле убийстве пытается убедить остал...,присяжн дел убийств пыта убед остальн присяжн ...,присяжный дело убийство пытаться убедить остал...,0


In [32]:
df['cluster'].value_counts()

cluster
2    66
3    60
0    43
4    41
1    39
Name: count, dtype: int64

In [33]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [34]:
df[df['cluster']==0]

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
2,Темный рыцарь,Бэтман вершит правосудие в Готэме. Его партнер...,12.08.2008,"боевик, драма, кинокомикс, криминал, триллер",9.12/10,бэтман вершит правосудие в готэме его партнера...,бэтман вершит правосудие готэме партнерами ста...,бэтма верш правосуд готэм партнер станов инспе...,бэтман вершить правосудие готэма партнёр стано...,0
4,12 разгневанных мужчин,Один из 12 присяжных в деле об убийстве пытае...,29.07.1957,"детектив, драма",9.15/10,один из присяжных в деле об убийстве пытается ...,присяжных деле убийстве пытается убедить остал...,присяжн дел убийств пыта убед остальн присяжн ...,присяжный дело убийство пытаться убедить остал...,0
9,"Хороший, плохой, злой",Действие происходит в разгар гражданской войны...,23.12.1966,"боевик, вестерн, приключения",8.96/10,действие происходит в разгар гражданской войны...,действие происходит разгар гражданской войны а...,действ происход разгар гражданск войн америк о...,действие происходить разгар гражданский война ...,0
23,Спасти рядового Райана,Вторая Мировая Война. Одна женщина теряет трои...,24.07.1998,"боевик, военный, драма",8.94/10,вторая мировая война одна женщина теряет троих...,вторая мировая война одна женщина теряет троих...,втор миров войн одн женщин теря тро сво сынов ...,второй мировой война женщина терять трое сын ч...,0
25,Жизнь прекрасна,Вторая мировая война. Италия.Фашисты арестовыв...,20.12.1997,"военный, драма, комедия, мелодрама",8.91/10,вторая мировая война италия фашисты арестовыва...,вторая мировая война италия фашисты арестовыва...,втор миров войн итал фашист арестовыва евре от...,второй мировой война италия фашист арестовыват...,0
27,Терминатор 2: Судный день,Первая попытка убить Сару Коннор еще до рожден...,03.07.1991,"боевик, триллер, фантастика",9.41/10,первая попытка убить сару коннор еще до рожден...,первая попытка убить сару коннор рождения сына...,перв попытк уб сар коннор рожден сын провал ск...,первый попытка убить сара коннора рождение сын...,0
34,Гладиатор,Великий генерал Максимус должен был стать насл...,04.05.2000,"боевик, драма, исторический, приключения",9.09/10,великий генерал максимус должен был стать насл...,великий генерал максимус должен стать наследни...,велик генера максимус долж стат наследник кеса...,великий генерал максимус должный стать наследн...,0
35,Король Лев,"На глаза у малыша Симбы погиб его отец, король...",02.07.1994,"детский, драма, комедия, мультфильм, мюзикл, п...",8.95/10,на глаза у малыша симбы погиб его отец король ...,глаза малыша симбы погиб отец король лев муфас...,глаз малыш симб погиб отец корол лев муфас спа...,глаз малыш симб погибнуть отец король лев муфа...,0
49,Новые времена,Легендарный Чарли Чаплин играет роль Маленьког...,11.02.1936,"драма, комедия, мелодрама",9.00/10,легендарный чарли чаплин играет роль маленьког...,легендарный чарли чаплин играет роль маленьког...,легендарн чарл чаплин игра рол маленьк бродяг ...,легендарный чарли чаплин играть роль маленький...,0
58,Индиана Джонс: В поисках утраченного ковчега,Индиана Джонс - известный археолог и специалис...,12.06.1981,"боевик, приключения, триллер",8.83/10,индиана джонс известный археолог и специалист ...,индиана джонс известный археолог специалист ок...,индиа джонс известн археолог специалист оккуль...,индиана джонс известный археолог специалист ок...,0


In [35]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=100000990000000000000,
                                 min_df=0.01, stop_words=russian_stopwords,
                                 ngram_range=(1,3))
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==0]['text_lemm'])
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: первый, человек, найти, друг, джонс, убить, коннора, будущее, сын, дело
Topic 1: дело, убийство, конец, решать, война, расследование, дочь, расследование дело, присяжный, присяжный дело
Topic 2: дело, убийство, присяжный дело, присяжный, конец, расследование дело, расследование, ён, остальной, полиция
Topic 3: чарли чаплин, чаплин, бродяга, чарли, золото, отправиться, приключение, чаплин отправиться, отправиться клондайк поиск, отправиться клондайк
Topic 4: джонс, индиана джонс, индиана, реликвия, археолог, опасный, рука, сша, дочь, вернуть
Topic 0: коннора, будущее, убить, человек, первый, сара, машина, убить сара коннора, убить сара, скайнуть
Topic 1: друг, сын, дочь, отправляться, отец, образ, человек, живой, найти, поиск
Topic 2: дело, убийство, конец, решать, расследование дело, война, присяжный дело, присяжный, женщина, расследование
Topic 3: чарли, бродяга, чарли чаплин, чаплин, золото, отправиться клондайк поиск, отправиться клондайк, поиск золото, чаплин отправиться, 

In [36]:
df[df['cluster']==1]

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
7,Криминальное чтиво,Три истории из жизни двух бандитов - Винсента ...,23.09.1994,"драма, криминал",8.94/10,три истории из жизни двух бандитов винсента ве...,истории жизни двух бандитов винсента вега джул...,истор жизн двух бандит винсент вег джулс винфи...,история жизнь бандит винсент вега джулс винфил...,1
10,Форрест Гамп,Форресту Гампу - умственно отсталому человеку ...,23.06.1994,"драма, комедия, мелодрама",9.40/10,форресту гампу умственно отсталому человеку с ...,форресту гампу умственно отсталому человеку до...,форрест гамп умствен отстал человек добр сердц...,форрест гампа умственно отсталый человек добры...,1
12,Бойцовский клуб,"Обычный клерк, угнетенный своей обыденной жизн...",15.10.2009,"драма, криминал, триллер",8.90/10,обычный клерк угнетенный своей обыденной жизнь...,обычный клерк угнетенный своей обыденной жизнь...,обычн клерк угнетен сво обыден жизн торговец с...,обычный клерк угнести обыденный жизнь торговец...,1
15,Матрица,"Он жил своими серыми буднями, он был очень тал...",31.03.1999,"боевик, приключения, триллер, фантастика",9.20/10,он жил своими серыми буднями он был очень тала...,жил своими серыми буднями очень талантливым пр...,жил сво сер будн очен талантлив программист по...,жить серый будни очень талантливый программист...,1
20,Эта замечательная жизнь,Ангел помогает положительному во всех отношени...,07.01.1947,"драма, мелодрама, фэнтези",8.35/10,ангел помогает положительному во всех отношени...,ангел помогает положительному отношениях бизне...,ангел помога положительн отношен бизнесмен око...,ангел помогать положительный отношение бизнесм...,1
29,Назад в будущее,"Марти был обычным выпускником, пока его ""сумас...",15.08.1985,"боевик, комедия, приключения, фантастика",9.15/10,марти был обычным выпускником пока его сумасше...,марти обычным выпускником пока сумасшедший дру...,март обычн выпускник пок сумасшедш друг профес...,марти обычный выпускник пока сумасшедший друг ...,1
33,Психо,"Мэрион Крейн похищает 40 тысяч долларов, сбега...",25.08.1960,"детектив, триллер, ужасы",8.70/10,мэрион крейн похищает тысяч долларов сбегает и...,мэрион крейн похищает тысяч долларов сбегает г...,мэрион крейн похища тысяч доллар сбега город о...,мэрион крейн похищать тысяча доллар сбегать го...,1
36,Человек-паук: Паутина вселенных,Продолжение истории о подростке из Нью-Йорка М...,01.06.2023,"боевик, кинокомикс, мультфильм, приключения, с...",8.60/10,продолжение истории о подростке из нью йорка м...,продолжение истории подростке нью йорка майлзе...,продолжен истор подростк нью йорк майлз морале...,продолжение история подросток нью йорк майлз м...,1
37,Отступники,Полицейские внедряют в окружение мафиози Фрэнк...,26.09.2006,"драма, криминал, триллер",8.80/10,полицейские внедряют в окружение мафиози фрэнк...,полицейские внедряют окружение мафиози фрэнка ...,полицейск внедря окружен мафиоз фрэнк костелл ...,полицейский внедрять окружение мафиози фрэнк к...,1
45,Подозрительные лица,"Полиция, занимаясь расследованием причин взрыв...",19.07.1995,"детектив, криминал, триллер",8.32/10,полиция занимаясь расследованием причин взрыва...,полиция занимаясь расследованием причин взрыва...,полиц заним расследован причин взрыв пирс сан ...,полиция заниматься расследование причина взрыв...,1


In [37]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=100000990000000000000,
                                 min_df=0.01, stop_words=russian_stopwords,
                                 ngram_range=(1,3))
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==1]['text_lemm'])
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: жизнь, человек, жена, история, обычный, работа, миллион, ребёнок, друг, совершенно
Topic 1: работа, го, го век, век, агент, рикки, рабство, меррик, джозеф, принять
Topic 2: жена, рабство, джанго, становиться, зритель, век, го, го век, доктор, находиться
Topic 3: жена, стать, мужчина, лицо, заветный мечта, реальность конец жизнь, мультфильм заветный, мультфильм, мультфильм заветный мечта, заветный мечта приключение
Topic 4: становиться, ночь, мэрион, отец, пока, мир, обычный, тысяча, ребёнок, весь
Topic 0: друг, человек, обычный, жизнь, пока, игрушка, приключение, остаться, марти, отец
Topic 1: полицейский, работа, грабитель, костелло, принять, анджелес, лос анджелес, лос, фрэнк, работать
Topic 2: жена, рабство, джанго, жестокий, становиться, уходить, леонард, предложение, бывший, пытаться
Topic 3: нью йорк, нью, мочь, йорк, взрыв, доллар, история, человек, миллион доллар, год
Topic 4: жизнь, семья, младший, младший брат, брат, мальчик, следить, воспоминание, помогать, однажды


In [38]:
df[df['cluster']==2]

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
0,Побег из Шоушенка,"""Побег из Шоушенка"" - фильм, который считается...",06.01.1995,драма,9.44/10,побег из шоушенка фильм который считается одн...,побег шоушенка считается одним лучших истории ...,побег шоушенк счита одн лучш истор кин режиссе...,побег шоушенка считаться хороший история кино ...,2
6,Властелин колец 3: Возвращение Короля,Близится последняя битва за Средиземье. На пут...,17.12.2003,"боевик, приключения, фэнтези",9.48/10,близится последняя битва за средиземье на пути...,близится последняя битва средиземье пути войск...,близ последн битв средизем пут войск саурон ст...,близиться последний битва средиземье путь войс...,2
11,Властелин колец 2: Две крепости,Братство кольца распалось. Фродо и Сэм продолж...,05.12.2002,"боевик, приключения, фэнтези",9.39/10,братство кольца распалось фродо и сэм продолжа...,братство кольца распалось фродо сэм продолжают...,братств кольц распа фрод сэм продолжа пут морд...,братство кольцо распасться фродый сэм продолжа...,2
13,Начало,"В мире будущего, где существуют технологии, по...",14.07.2010,"боевик, триллер, фантастика",9.04/10,в мире будущего где существуют технологии позв...,мире будущего существуют технологии позволяющи...,мир будущ существ технолог позволя проника под...,мир будущее существовать технология позволять ...,2
16,Славные парни,Начинающий молодой гангстер Генри Хилл со свои...,12.09.1990,"биографический, драма, криминал",8.74/10,начинающий молодой гангстер генри хилл со свои...,начинающий молодой гангстер генри хилл своими ...,начина молод гангстер генр хилл сво двум закад...,начинающий молодой гангстер генри хилла закады...,2
17,Пролетая над гнездом кукушки,Один бродяга решил отдохнуть за государственны...,19.11.1975,драма,9.08/10,один бродяга решил отдохнуть за государственны...,бродяга решил отдохнуть государственный счет с...,бродяг реш отдохнут государствен счет симулиру...,бродяга решить отдохнуть государственный счёт ...,2
22,Молчание ягнят,"Клариссе Стерлинг, молодому агенту ФБР, поруче...",14.03.1991,"детектив, драма, криминал, триллер",9.08/10,клариссе стерлинг молодому агенту фбр поручено...,клариссе стерлинг молодому агенту фбр поручено...,кларисс стерлинг молод агент фбр поруч расслед...,кларисса стерлинг молодой агент фбр поручить р...,2
26,Зеленая миля,Чернокожий осужденный Джон Коффи (Майкл Кларк ...,10.12.1999,"драма, криминал, фэнтези",9.18/10,чернокожий осужденный джон коффи майкл кларк д...,чернокожий осужденный джон коффи майкл кларк д...,чернокож осужден джон кофф майкл кларк дунка у...,чернокожий осудить джон коффи майкл кларк дунк...,2
31,Пианист,"Владислав Шпильман - известный музыкант, всеми...",25.09.2002,"биографический, военный, драма, музыка",8.72/10,владислав шпильман известный музыкант всеми си...,владислав шпильман известный музыкант всеми си...,владисла шпильма известн музыкант всем сил пыт...,владислав шпильман известный музыкант весь сил...,2
38,Одержимость,"История о молодом музыканте, преодолевающем ра...",13.11.2014,"драма, музыка",8.78/10,история о молодом музыканте преодолевающем раз...,история молодом музыканте преодолевающем разли...,истор молод музыкант преодолева различн препят...,история молодой музыкант преодолевать различны...,2


In [39]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=100000990000000000000,
                                 min_df=0.01, stop_words=russian_stopwords,
                                 ngram_range=(1,3))
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==2]['text_lemm'])
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: стать, год, хороший, жизнь, история, время, война, мировой, сын, зритель
Topic 1: последний, мать, вскоре, поиск, странный, деньга, барри, друг, великий, джерри
Topic 2: время великий депрессия, великий депрессия, время великий, великий, депрессия, чернокожий, защищать ложный обвинить, депрессия адвокат алабама, алабама аттикус, алабама аттикус финч
Topic 3: снятой мотив, мотив, снятой, баум, знаменитый произведение, мотив знаменитый произведение, снятой мотив знаменитый, мотив знаменитый, произведение фрэнк баум, произведение фрэнк
Topic 4: стать, ребёнок, великий депрессия, время великий депрессия, депрессия, время великий, великий, подобный, сальери, моцарт
Topic 0: хороший, кинематограф, история, стать, зритель, музыкальный, также, шоушенка, побег, побег шоушенка
Topic 1: ребёнок, мать, сын, девушка, комната, год, детский, джек, жить, вскоре
Topic 2: время великий, великий депрессия, время великий депрессия, великий, депрессия, время, защищать, америка время, защищать ложн

In [40]:
df[df['cluster']==3]

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
5,Список Шиндлера,Немецкий фабрикант и член нацистской партии Ос...,30.11.1993,"биографический, военный, драма, исторический",9.19/10,немецкий фабрикант и член нацистской партии ос...,немецкий фабрикант член нацистской партии оска...,немецк фабрикант член нацистск парт оскар шинд...,немецкий фабрикант член нацистский партия оска...,3
8,Властелин колец: Братство кольца,Поиски великого кольца Саурона завершены. Оно ...,19.12.2001,"боевик, приключения, фэнтези",9.33/10,поиски великого кольца саурона завершены оно п...,поиски великого кольца саурона завершены оно п...,поиск велик кольц саурон заверш он попа рук хо...,поиск великий кольцо саурон завершить оно попа...,3
14,Звездные войны: Эпизод 5 - Империя наносит отв...,В Галактике идет война. Имперские войска наход...,21.05.1980,"боевик, приключения, фантастика, фэнтези",8.98/10,в галактике идет война имперские войска находя...,галактике идет война имперские войска находят ...,галактик идет войн имперск войск наход баз пов...,галактика идти война имперский войско находить...,3
19,Семь,Двое полицейских расследуют дело серийного уби...,22.09.1995,"детектив, драма, криминал, триллер",8.88/10,двое полицейских расследуют дело серийного уби...,двое полицейских расследуют дело серийного уби...,дво полицейск расслед дел серийн убийц действ ...,двое полицейский расследовать дело серийный уб...,3
21,Семь самураев,"Исторический фильм, действие которого происход...",26.04.1954,"боевик, драма, приключения",8.97/10,исторический фильм действие которого происходи...,исторический действие которого происходит сред...,историческ действ котор происход средневеков я...,исторический действие происходить средневековы...,3
24,Город Бога,Город Бога - нищий квартал Рио-де-Жанейро. Луч...,30.08.2002,"драма, криминал, триллер",8.41/10,город бога нищий квартал рио де жанейро лучше ...,город бога нищий квартал рио де жанейро появля...,город бог нищ кварта ри де жанейр появля взрос...,город бог нищий квартал рио де жанейро появлят...,3
30,Унесенные призраками,Маленькая Тихиро вместе с мамой и папой переез...,20.07.2001,"аниме, детектив, мультфильм, приключения, семе...",8.77/10,маленькая тихиро вместе с мамой и папой переез...,маленькая тихиро вместе мамой папой переезжают...,маленьк тихир вмест мам пап переезжа нов дом з...,маленький тихиро вместе мама папа переезжать н...,3
41,Могила светлячков,Трагическая история о маленьком мальчике и его...,16.04.1988,"аниме, военный, драма, мультфильм",8.61/10,трагическая история о маленьком мальчике и его...,трагическая история маленьком мальчике младшей...,трагическ истор маленьк мальчик младш сестренк...,трагический история маленький мальчик младший ...,3
42,Престиж,В этой вольной экранизации романа Кристофера П...,17.10.2006,"детектив, драма, триллер, фантастика, фэнтези",8.79/10,в этой вольной экранизации романа кристофера п...,вольной экранизации романа кристофера приста р...,вольн экранизац рома кристофер прист реч идет ...,вольный экранизация роман кристофер пристый ре...,3
43,Харакири,Хансиро Цугумо когда-то был самураем в клане Ф...,16.09.1962,"боевик, драма, исторический",9.67/10,хансиро цугумо когда то был самураем в клане ф...,хансиро цугумо самураем клане фокушима дом пот...,хансир цугум самура клан фокушим дом потерпел ...,хансиро цугумо самурай клан фокушить дом потер...,3


In [41]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=100000990000000000000,
                                 min_df=0.01, stop_words=russian_stopwords,
                                 ngram_range=(1,3))
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==3]['text_lemm'])
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: год, война, мировой война, второй, второй мировой война, второй мировой, мировой, американский, еврей, время
Topic 1: город, девочка, маленький девочка, полицейский, каждый, год, жить, маленький, день, найти
Topic 2: девочка, маленький, маленький девочка, голова, формирование мысль голова, мысль, голова маленький, мысль голова маленький, мысль голова, мультфильм
Topic 3: американский, год американский, год, имя, городок, молодой, молодой человек, городок париж, потерять год, имя трэвиснуть потерять
Topic 4: планета, война, делать, человек, ива, валл, кольцо, жить, молодой, молодой человек
Topic 0: второй, мировой, второй мировой война, второй мировой, мировой война, война, еврей, американский, время второй, время второй мировой
Topic 1: девочка, город, маленький, маленький девочка, жить, полицейский, офелия, рассказать, голова, ребёнок
Topic 2: день, век, каждый, го век, го, судьба, пёс, революция го, кровавый, кровавый революция
Topic 3: американский, год, год американский, и

In [42]:
df[df['cluster']==4]

,title,description,release_date,tags,rating,prep_text,tokenize_text,text_stem,text_lemm,cluster
1,Крестный отец,"Фильм, ставший классикой своего жанра, рассказ...",24.03.1972,"драма, триллер",9.11/10,фильм ставший классикой своего жанра рассказыв...,ставший классикой своего жанра рассказывающий ...,ставш классик сво жанр рассказыва истор влияте...,стать классика жанр рассказывать история влият...,4
3,Крестный отец 2,Продолжение эпохальной саги режиссера Френсиса...,12.12.1974,"драма, криминал",8.88/10,продолжение эпохальной саги режиссера френсиса...,продолжение эпохальной саги режиссера френсиса...,продолжен эпохальн саг режиссер френсис форд к...,продолжение эпохальный сага режиссёр френсиса ...,4
18,Интерстеллар,Земля изживает себя и человечество должно иска...,06.11.2014,"детектив, приключения, фантастика",8.71/10,земля изживает себя и человечество должно иска...,земля изживает человечество должно искать новы...,земл изжива человечеств должн иска нов дом эт ...,земля изживать человечество должный искать нов...,4
28,Звездные войны: Эпизод 4 - Новая надежда,Принцесса Лейя - предводительница повстанцев з...,25.05.1977,"боевик, приключения, фантастика, фэнтези",8.92/10,принцесса лейя предводительница повстанцев зах...,принцесса лейя предводительница повстанцев зах...,принцесс лей предводительниц повстанц захвач з...,принцесса лейя предводительница повстанец захв...,4
32,Паразиты,Семья безработных не желает трудиться и живут ...,21.05.2019,"драма, комедия, триллер",8.28/10,семья безработных не желает трудиться и живут ...,семья безработных желает трудиться живут счет ...,сем безработн жела труд живут счет шантаж бога...,семья безработный желать трудиться жить счёт ш...,4
39,Американская история Х,"Дерек Виньярд вернулся из тюрьмы, чтобы найти ...",12.02.1999,"драма, криминал",8.58/10,дерек виньярд вернулся из тюрьмы чтобы найти с...,дерек виньярд вернулся тюрьмы найти своего мла...,дерек виньярд вернул тюрьм найт сво младш брат...,дерек виньярд вернуться тюрьма найти младший б...,4
40,Леон,Леон - профессиональный убийца. Он с легкостью...,14.09.1994,"боевик, драма, криминал, триллер",8.95/10,леон профессиональный убийца он с легкостью вы...,леон профессиональный убийца легкостью выполня...,леон профессиональн убийц легкост выполня сам ...,леон профессиональный убийца лёгкость выполнят...,4
47,1+1,Основанная на реальных событиях история необыч...,02.11.2011,"драма, комедия",8.99/10,основанная на реальных событиях история необыч...,основанная реальных событиях история необычной...,основа реальн событ истор необычн дружб двух с...,основать реальный событие история необычный др...,4
52,Однажды на Диком Западе,Не расстающийся со совей губной гармоникой таи...,21.12.1968,"вестерн, драма",8.48/10,не расстающийся со совей губной гармоникой таи...,расстающийся совей губной гармоникой таинствен...,расста сов губн гармоник таинствен бродяг изве...,расставаться советь губный гармоника таинствен...,4
53,Огни большого города,Маленький бродяга встречает на улице слепую цв...,27.02.1931,"драма, комедия, мелодрама",9.00/10,маленький бродяга встречает на улице слепую цв...,маленький бродяга встречает улице слепую цвето...,маленьк бродяг встреча улиц слеп цветочниц зад...,маленький бродяга встречать улица слепой цвето...,4


In [43]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=100000990000000000000,
                                 min_df=0.01, stop_words=russian_stopwords,
                                 ngram_range=(1,3))
tfidf_matrix = tfidf_vectorizer.fit_transform(df[df['cluster']==1]['text_lemm'])
# создание модели LSA
lsa_model = TruncatedSVD(n_components=5, random_state=0)
lsa_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(lsa_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: жизнь, человек, жена, история, обычный, работа, миллион, ребёнок, друг, совершенно
Topic 1: работа, го, го век, век, агент, рикки, рабство, меррик, джозеф, принять
Topic 2: жена, рабство, джанго, становиться, зритель, век, го, го век, доктор, находиться
Topic 3: жена, стать, мужчина, лицо, заветный мечта, реальность конец жизнь, мультфильм заветный, мультфильм, мультфильм заветный мечта, заветный мечта приключение
Topic 4: становиться, ночь, мэрион, отец, пока, мир, обычный, тысяча, ребёнок, весь
Topic 0: друг, человек, обычный, жизнь, пока, игрушка, приключение, остаться, марти, отец
Topic 1: полицейский, работа, грабитель, костелло, принять, анджелес, лос анджелес, лос, фрэнк, работать
Topic 2: жена, рабство, джанго, жестокий, становиться, уходить, леонард, предложение, бывший, пытаться
Topic 3: нью йорк, нью, мочь, йорк, взрыв, доллар, история, человек, миллион доллар, год
Topic 4: жизнь, семья, младший, младший брат, брат, мальчик, следить, воспоминание, помогать, однажды


In [44]:
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

## Классификация

In [45]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['tokenize_text'], df['cluster'], 
                                                      test_size=0.3, 
                                                      random_state=0)

In [46]:
len(X_train)

174

In [47]:
len(X_test)

75

In [48]:
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [49]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3), stop_words=russian_stopwords)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## LogisticRegression

In [50]:
model_lr = LogisticRegression()
model_lr.fit(X_train_tfidf, y_train)

LogisticRegression()

In [51]:
y_pred = model_lr.predict(X_test_tfidf)

In [52]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.00      0.00      0.00        16
           2       0.33      0.95      0.49        20
           3       0.44      0.47      0.46        17
           4       0.00      0.00      0.00        12

    accuracy                           0.36        75
   macro avg       0.16      0.28      0.19        75
weighted avg       0.19      0.36      0.24        75



In [53]:
X_test_tfidf.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [54]:
y_pred

array([2, 2, 2, 2, 3, 2, 2, 3, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 3, 2,
       2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 3, 2, 2, 3, 3, 2, 3, 2, 3, 2, 2, 3,
       3, 2, 2, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 3, 2, 2, 2,
       2, 2, 2, 3, 2, 2, 2, 2, 2], dtype=int64)

## RandomForestClassifier

In [55]:
model_rf = RandomForestClassifier()
model_rf.fit(X_train_tfidf, y_train)

RandomForestClassifier()

In [56]:
y_pred = model_rf.predict(X_test_tfidf)

In [57]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        10
           1       0.00      0.00      0.00        16
           2       0.29      0.80      0.42        20
           3       0.35      0.35      0.35        17
           4       0.00      0.00      0.00        12

    accuracy                           0.29        75
   macro avg       0.13      0.23      0.15        75
weighted avg       0.16      0.29      0.19        75



## KNeighborsClassifier

In [58]:
model_knn = KNeighborsClassifier()
model_knn.fit(X_train_tfidf, y_train)

KNeighborsClassifier()

In [59]:
y_pred = model_knn.predict(X_test_tfidf)

In [60]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.18      0.30      0.22        10
           1       0.36      0.25      0.30        16
           2       0.34      0.50      0.41        20
           3       0.45      0.29      0.36        17
           4       0.29      0.17      0.21        12

    accuracy                           0.32        75
   macro avg       0.33      0.30      0.30        75
weighted avg       0.34      0.32      0.32        75



## Сохранение модели

In [61]:
import pickle

with open('film_model.pkl', 'wb') as f:
    pickle.dump(model_knn, f)

In [62]:
with open('film_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

In [63]:
df.to_csv('data_movies.csv')

In [64]:
t1 = "Бухгалтер Энди Дюфрейн обвинён в убийстве собственной жены и её любовника. Оказавшись в тюрьме под названием Шоушенк; он сталкивается с жестокостью и беззаконием; царящими по обе стороны решётки. Каждый; кто попадает в эти стены; становится их рабом до конца жизни. Но Энди; обладающий живым умом и доброй душой; находит подход как к заключённым; так и к охранникам; добиваясь их особого к себе расположения."

In [65]:
with open('film_model.pkl', 'rb') as file:
    model = pickle.load(file)

with open('film_vectorizer.pkl', 'rb') as file:
    vectorizer = pickle.load(file)

In [66]:
def fun_punctuation_text(text):
    text = text.lower()
    text = ''.join([ch for ch in text if ch not in string.punctuation])
    text = ''.join([i if not i.isdigit() else '' for i in text])
    text = ''.join([i if i.isalpha() else ' ' for i in text])
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    text = re.sub('[a-z]', '', text, flags=re.I)
    st = '❯\xa0'
    text = ''.join([ch if ch not in st else ' ' for ch in text])
    return text

In [67]:
def fun_lemmatizing_text(text):
    tokens = word_tokenize(text)
    res = list()
    for word in tokens:
        p = pymorphy3.MorphAnalyzer(lang='ru').parse(word)[0]
        res.append(p.normal_form)  
    text = " ".join(res)
    return text

In [68]:
def fun_tokenize(text):
    t = word_tokenize(text)
    tokens = [token for token in t if token not in russian_stopwords]
    text = " ".join(tokens)
    return text

In [ ]:
t1 = fun_tokenize(fun_lemmatizing_text(fun_punctuation_text(t1)))

In [ ]:
text_vectorized = vectorizer.transform([t1])

In [ ]:
text_vectorized.toarray()

In [ ]:
prediction = model.predict(text_vectorized)  
probabilities = model.predict_proba(text_vectorized)

In [ ]:
print(f"Класс: {prediction[0]}")
print(f"Вероятности: {probabilities}")

In [ ]:
t2 = "Влиятельный гангстер из Лондона решает продать свой бизнес по справедливой цене. Но не все в его сфере уважают справедливость. Бандиты из разных частей света начинают охоту на «стареющего льва». Британский король криминального жанра, культовый режиссер Гай Ричи представляет лихо закрученную, ироничную и неполиткорректную гангстерскую комедию в лучших традициях своего стиля и юмора. Колоритных и опасных «джентльменов» сыграли Мэттью МакКонахи, Чарли Ханнэм, Хью Грант и Колин Фаррелл, каким вы его еще не видели. Амбициозный кинодраматург (а вообще-то – беспринципный шантажист и скользкий частный сыщик) Флетчер рассказывает сюжет своего будущего фильма на основе реальных событий. Это история большого человека по имени Микки Пирсон, который с нуля создал многомиллионный бизнес по выращиванию и сбору нелегального «урожая». Когда его лихая молодость осталась позади, он решил стать примерным семьянином, передав дела надежному человеку. Желающих более чем достаточно, но появляется камень преткновения – цена. А также – принципиальность Пирсона, не идущего на уступки. Враги и предатели за его спиной начинают плести интриги, его правая рука Реймонд пытается держать оборону, а тем временем пронырливый наблюдатель Флетчер собирает компромат. Но даже он не видит всей картины и не подозревает, какая готовится кульминация."

In [ ]:
t2 = fun_tokenize(fun_lemmatizing_text(fun_punctuation_text(t2)))

In [ ]:
text_vectorized = vectorizer.transform([t2])

In [ ]:
text_vectorized.toarray()

In [ ]:
prediction = model.predict(text_vectorized)  
probabilities = model.predict_proba(text_vectorized)

In [ ]:
print(f"Класс: {prediction[0]}")
print(f"Вероятности: {probabilities}")